# SecureFace-RX — 저장된 보호본 복원 (코랩 GPU)

라즈베리파이가 저장한 PSF 보호본 청크를 **코랩 GPU**에서 INN 역변환으로 복원합니다.
라즈베리파이 CPU보다 훨씬 빠르게 모든 프레임을 복원해 실제 시간 길이 영상을 만듭니다.

**흐름**: 라즈베리파이가 보호본 저장 → 청크를 압축해 업로드 → GPU 복원 → 다운로드

**실행 전**: 런타임 → 런타임 유형 변경 → T4 GPU

## 1. GPU 확인 + 코드 받기

In [ ]:
!nvidia-smi -L
!git clone https://github.com/ins420/scrfd-proface.git
%cd scrfd-proface
!git pull
import torch; print('CUDA:', torch.cuda.is_available())

## 2. 패키지 설치

In [ ]:
!pip install -q opencv-python-headless pycryptodome
print('설치 완료')

## 3. INN 체크포인트(.pth) 업로드

In [ ]:
from google.colab import files
import shutil, os
os.makedirs('checkpoints', exist_ok=True)
print('▶ .pth 파일을 선택하세요')
up = files.upload()
for fn in up:
    if fn.endswith('.pth'):
        shutil.move(fn, f'checkpoints/{fn}')
        print(f'  {fn} → checkpoints/')
import config as c
print('INN_CHECKPOINT =', c.INN_CHECKPOINT, '| 존재:', os.path.exists(c.INN_CHECKPOINT))

## 4. 보호본 청크 업로드
라즈베리파이에서 청크를 압축해 만든 `chunk.tar.gz` 를 업로드하세요.

라즈베리파이에서 압축하는 법 (예: 14시 14-00 청크):
```bash
cd ~/scrfd-proface
tar czf chunk.tar.gz recordings/2026-06/29/오후/14시/14-00
```
그 `chunk.tar.gz` 를 PC로 옮겨(scp) 여기에 업로드.

In [ ]:
from google.colab import files
import tarfile, os
print('▶ chunk.tar.gz 를 선택하세요')
up = files.upload()
tar_name = next(iter(up))
with tarfile.open(tar_name) as t:
    t.extractall('.')
print('압축 해제 완료')
# 압축 안에 있는 청크 폴더(가장 안쪽, meta.json/frame.jpg 있는 상위) 자동 탐색
chunk_dir = None
for root, dirs, fs in os.walk('recordings'):
    if any(d.isdigit() for d in dirs):
        chunk_dir = root
        break
print('청크 폴더:', chunk_dir)

## 5. GPU로 복원 → 영상 생성

In [ ]:
from restore_chunk import restore_chunk
out = restore_chunk(chunk_dir, password='forensic2026', out_path='restored.mp4')
print('결과:', out)

## 6. 미리보기 + 다운로드

In [ ]:
from IPython.display import HTML
from base64 import b64encode
mp4 = open('restored.mp4', 'rb').read()
HTML(f'<video width=720 controls><source src="data:video/mp4;base64,{b64encode(mp4).decode()}" type="video/mp4"></video>')

In [ ]:
from google.colab import files
files.download('restored.mp4')